# 04_sequential_agent

04_sequential_agent.py — SequentialAgent: 순차 워크플로우

핵심: `output_key` 로 한 에이전트의 출력을 State 에 저장하고,
다음 에이전트가 `instruction` 의 {키} 로 그 값을 참조.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 이미 이벤트 루프가 돌아 스크립트의 asyncio.run() 이 깨짐 → nest_asyncio 로 중첩 허용
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '04_sequential_agent.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
04_sequential_agent.py — SequentialAgent: 순차 워크플로우

핵심: `output_key` 로 한 에이전트의 출력을 State 에 저장하고,
다음 에이전트가 `instruction` 의 {키} 로 그 값을 참조.
"""
from google.adk.agents import LlmAgent, SequentialAgent

from _adk_common import adk_model, run_once, banner, adk_unavailable


def main() -> None:
    banner("SequentialAgent — 조사 → 작성 파이프라인")

    researcher = LlmAgent(
        name="researcher",
        model=adk_model(),
        instruction=(
            "다음 주제에 대해 핵심 사실 2 가지를 한 줄씩 정리하라. "
            "주제를 그대로 반복하지 말고 사실만 출력."
        ),
        output_key="research",        # 출력 → state["research"]
    )

    writer = LlmAgent(
        name="writer",
        model=adk_model(),
        instruction=(
            "다음 조사 결과를 바탕으로 한국어로 짧은 한 단락을 작성하라:\n\n"
            "조사 결과:\n{research}"   # state["research"] 참조
        ),
        output_key="article",
    )

    pipeline = SequentialAgent(
        name="research_pipeline",
        sub_agents=[researcher, writer],
        description="조사 후 글을 작성하는 순차 파이프라인",
    )

    try:
        reply = run_once(pipeline, "LangGraph 의 핵심 특징")
        print(f"\n  ❓ topic : LangGraph 의 핵심 특징")
        print(f"\n  💬 article (writer 출력)")
        print(f"  {reply[:400]}")
        print(f"\n  💡 출력은 state['article'] 에 저장됨 — 더 긴 파이프라인 가능")
    except Exception as e:
        print(f"\n  ⚠ {type(e).__name__}: {str(e)[:200]}")
        adk_unavailable()


if __name__ == "__main__":
    main()


📌 SequentialAgent — 조사 → 작성 파이프라인


C:\Users\user\AppData\Local\Temp\ipykernel_35236\1457588217.py:35: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  pipeline = SequentialAgent(



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers




  ❓ topic : LangGraph 의 핵심 특징

  💬 article (writer 출력)
  상태(State)를 유지하며 순환 구조(Cycles)가 포함된 복잡한 워크플로우를 설계할 수 있습니다.
에이전트의 작업 흐름을 그래프 단위로 구조화하여 정교한 제어와 확장이 가능합니다.

  💡 출력은 state['article'] 에 저장됨 — 더 긴 파이프라인 가능
